In [88]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from pyarrow import dictionary

from func.UsefulFunctions import *

In [89]:
groupDF = pl.read_parquet("Data/pipelinedGroupsWikidata.parquet")
groupDF.head()

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64
"""Q72092""","""Rage Against the Machine""","""Q877693""","""Q215380""",2001.0,"""Q30""","""Q216364""","""rap rock""","""musical group""","""United States""","""Epic Records""",null,null,"""3798b104-01cb-484c-a3b0-56adc6…","""https://en.wikipedia.org/wiki/…",125.0,0
"""Q72092""","""Rage Against the Machine""","""Q877693""","""Q215380""",2001.0,"""Q30""","""Q1759336""","""rap rock""","""musical group""","""United States""","""Revelation Records""",null,null,"""3798b104-01cb-484c-a3b0-56adc6…","""https://en.wikipedia.org/wiki/…",125.0,1
"""Q72349""","""Krewella""","""Q20474""","""Q14073567""",2010.0,"""Q30""","""Q183387""","""dubstep""","""sibling duo""","""United States""","""Columbia Records""",null,null,"""d87475fa-76a1-4e12-962c-fc9941…","""https://en.wikipedia.org/wiki/…",73.0,2
"""Q72349""","""Krewella""","""Q20474""","""Q14073567""",2010.0,"""Q30""","""Q10398667""","""dubstep""","""sibling duo""","""United States""","""Monstercat""",null,null,"""d87475fa-76a1-4e12-962c-fc9941…","""https://en.wikipedia.org/wiki/…",73.0,3
"""Q72349""","""Krewella""","""Q20502""","""Q14073567""",2010.0,"""Q30""","""Q183387""","""house music""","""sibling duo""","""United States""","""Columbia Records""",null,null,"""d87475fa-76a1-4e12-962c-fc9941…","""https://en.wikipedia.org/wiki/…",73.0,4


In [90]:
groupDF.schema

Schema([('group', String),
        ('groupLabel', String),
        ('genre', String),
        ('groupType', String),
        ('inception', Float64),
        ('countryOfOrigin', String),
        ('recordLabel', String),
        ('genreLabel', String),
        ('groupTypeLabel', String),
        ('countryOfOriginLabel', String),
        ('recordLabelLabel', String),
        ('influencedby', String),
        ('influencedbyLabel', String),
        ('musicBrainzID', String),
        ('article', String),
        ('statementCount', Float64),
        ('__index_level_0__', Int64)])

In [91]:
#Look at the size of the dataset
size = groupDF.select(pl.col("group")).unique()
print(f"There are {len(size)} groups in this dataset")

There are 125699 groups in this dataset


In [ ]:
#What are the unique group types
groupDF.select(
    pl.col(["groupType", "groupTypeLabel"])
).unique().write_csv("variousGroupTypes.csv")


In [92]:
#Drop everyone who is fictitious as no characters are allowed
#Get all the people who have have fictitious in there
characterEntities= groupDF.filter(
    pl.col("groupTypeLabel").str.contains("(?i)fictional")
).select(pl.col("group")).unique()

#drop everyone who is not fictional and deduplicate
groupDF = groupDF.filter(
    ~pl.col("group").is_in(characterEntities["group"].implode())
).unique()


print(f"There are {len(groupDF["group"].unique())} groups in this dataset")




There are 125650 groups in this dataset


In [74]:
def nullColumnLabelWithissues(df, column):
    columnLabel = column + "Label"

    noLabel = df.filter(
        pl.col(column).is_in(pl.col(columnLabel).implode())
    ).select(pl.col(columnLabel)).unique()
    noLabel_dict= {k: None for k in noLabel[columnLabel].to_list()}
    df = df.with_columns(pl.col(columnLabel).replace(noLabel_dict))

    nolabelDF = df.filter(
        pl.col(columnLabel).is_null(),
    ).select(pl.col(["group","groupLabel"])).unique()
    display(nolabelDF)
    print(f"There are {len(nolabelDF)} groups with no default labels/labels in english in Column: {columnLabel}.")
    if column == "genre":
        nolabelDF.write_csv(f"./missingValuesStore/groupEntitiesWithoutLabels.csv")
    else:
        nolabelDF.write_csv(f"./missingValuesStore/groupEntitiesWithoutLabelled{column}OrNo{column}.csv")

    return df


In [75]:
#Null the group name labels for the groups who do not have a group label
groupDF = nullColumnLabelWithissues(groupDF, "group")

group,groupLabel
str,str
"""Q106200620""",null
"""Q102171690""",null
"""Q2798566""",null
"""Q11346973""",null
"""Q85982040""",null
…,…
"""Q10520347""",null
"""Q13026103""",null
"""Q105638589""",null


There are 15927 groups with no default labels/labels in english in Column: groupLabel.


In [76]:
#Check the genres and the genreLabels - Cleaned previously in the pipline stage, to remove error ones
#No genre added had no name
groupDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
)
#create a list of artists with no labelled genres or no genres
entitiesWithNullGenres = groupDF.filter(
    pl.col("genreLabel").is_null()
).select(pl.col(["group","groupLabel"])).unique()
display(entitiesWithNullGenres)
entitiesWithNullGenres.write_csv("./missingValuesStore/groupEntitiesWithoutLabelledGenresOrNoGenres.csv")


group,groupLabel
str,str
"""Q1709757""","""Jot Drop"""
"""Q11180443""",null
"""Q82349074""",null
"""Q110895170""","""Orchestra Kingston"""
"""Q117352455""",null
…,…
"""Q77957261""","""Renato Carosone e il suo quart…"
"""Q12045703""","""Pláče kočka"""
"""Q120335679""",null


In [77]:
groupDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
)

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64


In [78]:
#Check the group Types. In pipeline stage already removed groupTypes that had no Q number from pipeline
groupDF = nullColumnLabelWithissues(groupDF, "groupType")

group,groupLabel
str,str
"""Q98969730""",null
"""Q112258309""","""1. Celler Handharmonika-Klub"""
"""Q12407543""",null
"""Q6606608""","""Meitar Ensemble"""
"""Q15825100""",null
…,…
"""Q91020216""",null
"""Q12085520""",null
"""Q24102017""","""Münchner Orchesterverein Wilde…"


There are 64 groups with no default labels/labels in english in Column: groupTypeLabel.


In [79]:
#Check the countryOfOrign. In pipeline stage already removed countryOfOrigin that had no Q number from earlier
groupDF = nullColumnLabelWithissues(groupDF, "countryOfOrigin")

group,groupLabel
str,str
"""Q135069828""","""Nukuluk"""
"""Q3071521""","""Fieldwork"""
"""Q2371241""","""Netherlands Youth Orchestra"""
"""Q110895170""","""Orchestra Kingston"""
"""Q25471283""",null
…,…
"""Q2304150""","""Garpa"""
"""Q11831718""","""Pulsarus"""
"""Q25303206""","""The Dearly Beloved"""


There are 34911 groups with no default labels/labels in english in Column: countryOfOriginLabel.


In [80]:
#Check the recordLabel and influencedby. As with the other fields the non-Q-numbers were removed
groupDF = nullColumnLabelWithissues(groupDF, "recordLabel")
groupDF = nullColumnLabelWithissues(groupDF, "influencedby")

group,groupLabel
str,str
"""Q12253272""","""Ama Say"""
"""Q17220437""","""Manikyua-Dan"""
"""Q4930539""","""Bluesberry Jam"""
"""Q6844118""","""Mighty Fairly"""
"""Q13740300""","""KAAL"""
…,…
"""Q10700097""","""Torgny Melins"""
"""Q16097141""","""Mokpo Symphony Orchestra"""
"""Q63352663""","""Desaiha"""


There are 101949 groups with no default labels/labels in english in Column: recordLabelLabel.


group,groupLabel
str,str
"""Q4048661""","""S.C.A.L.P."""
"""Q11224779""","""Indigo blue"""
"""Q7384900""","""Rydell & Quick"""
"""Q139738015""",null
"""Q12388006""","""Esenia"""
…,…
"""Q6784632""","""Masshysteri"""
"""Q1780184""","""Royal String Quartet (Warsaw, …"
"""Q38266606""","""Cacodaemon"""


There are 125699 groups with no default labels/labels in english in Column: influencedbyLabel.


In [81]:
#Look at the musicBrainz IDs to confirm that there isn't an issue observed, and it was fixed in the pipeline
groupDF.filter(
    (pl.col("musicBrainzID").str.contains("http")) | (pl.col("musicBrainzID").str.contains("user")) | (pl.col("musicBrainzID").str.contains("artist"))
)

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64


In [93]:
#Looked at the article column to confirm that the filters in the pipeline column worked correctly and only returned proper links
groupDF.filter(
    ~pl.col("article").str.contains("(?i)en.wikipedia")
)

group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,i64


In [83]:
#Change the inception column to years
groupsWithNoInception = groupDF.filter(
    pl.col("inception").is_null()
).select(['group','groupLabel']).unique()
print(f"There are {len(groupsWithNoInception)} groups with no inception year.")
groupsWithNoInception.write_csv("./missingValuesStore/groupEntitiesWithNoInceptionYear.csv")

There are 77002 groups with no inception year.
